# Ablation Study 3 — Identity Adjacency (Graph Structure vs. No Graph Structure)

**Question:** How much does knowing the bond graph actually help GCN? Is U0 mostly predictable from atomic composition alone, or does topology matter?  
**Model:** GCN only (via `GCNAblation` with `use_graph_structure` flag)  
**Variable:** `use_graph_structure` ∈ {True, False}  

### What the no-graph variant does
When `use_graph_structure=False`, `GCNConv` is replaced by `nn.Linear`. Each atom is updated **only from its own features** — no neighbor aggregation.
The architecture remains permutation-invariant and variable-size-safe through `global_mean_pool + global_max_pool`.
This is **not a standard MLP** — it is a node-wise shared-weight transform followed by invariant pooling.
The key question: does aggregating over bonded neighbors add information beyond atomic composition?

**Output directory:** `MyDrive/Ablation/Study3_IdentityAdjacency/`

**Required:** `models/gcn_ablation.py` must be present in your repo.

In [ ]:
# ── Cell 1: Mount Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────────
import torch, subprocess, sys
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
torch_version = torch.__version__.split('+')[0]
cuda_tag = 'cu121' if torch.cuda.is_available() else 'cpu'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-scatter', 'torch-sparse',
                '-f', f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric'], check=True)
print('PyG installed.')

In [ ]:
# ── Cell 3: Clone repo & set paths ────────────────────────────────────────────
import os, sys, subprocess

REPO_URL  = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'  # ← update this
REPO_DIR  = '/content/gnn_project'
DRIVE_OUT = '/content/drive/MyDrive/Ablation/Study3_IdentityAdjacency'
DATA_ROOT = '/content/qm9_data'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

for sub in ['checkpoints', 'logs', 'results', 'plots']:
    os.makedirs(f'{DRIVE_OUT}/{sub}', exist_ok=True)

print(f'Output root: {DRIVE_OUT}')

In [ ]:
# ── Cell 4: USER CONFIG ───────────────────────────────────────────────────────

# ---------- Training ----------
EPOCHS      = 100
PATIENCE    = 15
BATCH_SIZE  = 128
LR          = 5e-4

# ---------- Model ----------
HIDDEN_DIM   = 256
NUM_LAYERS   = 6      # use best from Study 1, or fix at 6
DROPOUT      = 0.0
FEATURE_MODE = 'topology'

# ---------- Ablation variable ----------
# True  = standard GCN (uses real bond graph)
# False = no-graph variant (node-wise Linear only, ignores bonds)
GRAPH_VARIANTS = [True, False]

# ---------- Dataset ----------
TARGET_IDX  = 7
SEED        = 42
SPLIT       = [0.8, 0.1, 0.1]

print('Config set.')

In [ ]:
# ── Cell 5: Load data ─────────────────────────────────────────────────────────
from data.loader import get_dataloaders

base_cfg = {
    'dataset':  {'target': TARGET_IDX, 'split': SPLIT, 'seed': SEED, 'feature_mode': FEATURE_MODE},
    'training': {'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'patience': PATIENCE, 'lr': LR},
}
train_loader, val_loader, test_loader, normalizer = get_dataloaders(base_cfg, root=DATA_ROOT)
print('Data loaded.')

In [ ]:
# ── Cell 6: Training functions ────────────────────────────────────────────────
import copy, csv, torch
import torch.nn.functional as F
from tqdm import tqdm
from data.features import select_features, get_feature_dims

# Import the ablation model directly (not via build_model factory)
from models.gcn_ablation import GCNAblation

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

def mae(pred, target):
    return (pred - target).abs().mean().item()


def run_epoch(model, loader, optimizer, device, normalizer, feature_mode, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_targets = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            batch = select_features(batch, mode=feature_mode)
            batch = batch.to(device)
            # GCNAblation always receives edge_index — ignored when use_graph_structure=False
            pred   = model(batch.x, batch.edge_index, batch.batch)
            target = batch.y.view(-1)
            loss   = F.mse_loss(pred, target)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * batch.num_graphs
            all_preds.append(normalizer.denormalize(pred.detach().cpu()))
            all_targets.append(normalizer.denormalize(target.detach().cpu()))
    n = sum(t.size(0) for t in all_targets)
    return total_loss / n, mae(torch.cat(all_preds), torch.cat(all_targets))


def train_run(use_graph_structure, run_id):
    feature_dims = get_feature_dims(FEATURE_MODE)
    model = GCNAblation(
        node_dim=feature_dims['node_dim'],
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        use_graph_structure=use_graph_structure,
    ).to(DEVICE)

    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    mode_label  = 'GCN+Graph' if use_graph_structure else 'GCN-NoGraph'
    print(f'  [{run_id}] {mode_label} | params={param_count:,}')

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    ckpt_path = f'{DRIVE_OUT}/checkpoints/{run_id}_best.pt'
    log_path  = f'{DRIVE_OUT}/logs/{run_id}_history.csv'

    best_val_mae, best_state, patience_ctr = float('inf'), None, 0
    history = []

    with open(log_path, 'w', newline='') as f:
        csv.DictWriter(f, fieldnames=['epoch','train_loss','val_loss','val_mae','lr']).writeheader()

    pbar = tqdm(range(1, EPOCHS + 1), desc=run_id, leave=True)
    for epoch in pbar:
        tr_loss, _ = run_epoch(model, train_loader, optimizer, DEVICE, normalizer, FEATURE_MODE, train=True)
        vl_loss, vl_mae_val = run_epoch(model, val_loader, None, DEVICE, normalizer, FEATURE_MODE, train=False)
        lr_now = optimizer.param_groups[0]['lr']
        row = {'epoch': epoch, 'train_loss': f'{tr_loss:.6f}', 'val_loss': f'{vl_loss:.6f}',
               'val_mae': f'{vl_mae_val:.6f}', 'lr': f'{lr_now:.2e}'}
        history.append(row)
        with open(log_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=['epoch','train_loss','val_loss','val_mae','lr']).writerow(row)
        pbar.set_postfix(val_mae=f'{vl_mae_val:.4f}')
        if vl_mae_val < best_val_mae:
            best_val_mae = vl_mae_val
            best_state   = copy.deepcopy(model.state_dict())
            patience_ctr = 0
            torch.save(best_state, ckpt_path)
        else:
            patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'  Early stop at epoch {epoch}')
            break

    print(f'  [{run_id}] best val MAE = {best_val_mae:.4f} Ha')
    return {'run_id': run_id, 'variant': mode_label, 'use_graph_structure': use_graph_structure,
            'best_val_mae': best_val_mae, 'params': param_count, 'checkpoint': ckpt_path}


print('Functions defined.')

In [ ]:
# ── Cell 7: Run both variants ─────────────────────────────────────────────────
import pandas as pd

results = []
results_path = f'{DRIVE_OUT}/results/study3_results.csv'

for use_graph in GRAPH_VARIANTS:
    run_id = 'gcn_graph' if use_graph else 'gcn_nograph'
    print(f'\nRunning {run_id} ...')
    result = train_run(use_graph, run_id)
    results.append(result)
    pd.DataFrame(results).to_csv(results_path, index=False)
    print(f'  Saved → {results_path}')

df = pd.DataFrame(results)
print('\n── Study 3 Results ──')
print(df[['variant', 'use_graph_structure', 'best_val_mae', 'params']].to_string(index=False))

# Print the delta clearly
graph_mae   = df[df['use_graph_structure'] == True]['best_val_mae'].values[0]
nograph_mae = df[df['use_graph_structure'] == False]['best_val_mae'].values[0]
delta = nograph_mae - graph_mae
pct   = delta / nograph_mae * 100
print(f'\nGCN+Graph:   {graph_mae:.4f} Ha')
print(f'GCN-NoGraph: {nograph_mae:.4f} Ha')
print(f'Delta (graph advantage): {delta:.4f} Ha ({pct:.1f}% relative improvement)')

In [ ]:
# ── Cell 8: CVPR-style bar chart + learning curves ────────────────────────────
import matplotlib, matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np, pandas as pd, os

matplotlib.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 9, 'axes.titlesize': 9, 'axes.labelsize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'figure.dpi': 300, 'axes.spines.top': False, 'axes.spines.right': False,
})

df = pd.read_csv(f'{DRIVE_OUT}/results/study3_results.csv')

fig, (ax_bar, ax_curve) = plt.subplots(1, 2, figsize=(3.5, 2.4))

# Left: bar comparison
variants = df['variant'].tolist()
maes     = df['best_val_mae'].tolist()
colors   = ['#648FFF', '#FFB000']
bars = ax_bar.bar(variants, maes, color=colors, edgecolor='white', linewidth=0.5, width=0.5)
for bar, v in zip(bars, maes):
    ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7)
ax_bar.set_ylabel('Val MAE (Ha)')
ax_bar.set_title('Topology vs. No Graph')
ax_bar.tick_params(axis='x', labelsize=7)

# Right: learning curves
run_ids = ['gcn_graph', 'gcn_nograph']
curve_colors = {'gcn_graph': '#648FFF', 'gcn_nograph': '#FFB000'}
curve_labels = {'gcn_graph': 'GCN+Graph', 'gcn_nograph': 'GCN-NoGraph'}
for run_id in run_ids:
    log_path = f'{DRIVE_OUT}/logs/{run_id}_history.csv'
    if not os.path.exists(log_path): continue
    hist = pd.read_csv(log_path)
    ax_curve.plot(hist['epoch'], hist['val_mae'].astype(float),
                  color=curve_colors[run_id], linewidth=1.2, label=curve_labels[run_id])
ax_curve.set_xlabel('Epoch')
ax_curve.set_ylabel('Val MAE (Ha)')
ax_curve.set_title('Learning Curves')
ax_curve.legend(frameon=False)

fig.tight_layout(pad=0.4)
plot_path = f'{DRIVE_OUT}/plots/study3_identity_adjacency.pdf'
fig.savefig(plot_path, format='pdf', bbox_inches='tight')
fig.savefig(plot_path.replace('.pdf', '.png'), format='png', bbox_inches='tight', dpi=300)
plt.show()
print(f'Plot saved → {plot_path}')